# Stochastic Volatility EDA
**Author:** Fidel Mehra  
**Project:** stochastic-volatility-mlops

This notebook performs exploratory data analysis (EDA) on market data to:
- Understand the distributional properties of log-returns
- Visualise rolling volatility regimes
- Inspect autocorrelation and volatility clustering
- Motivate the use of HMM for regime detection

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## 1. Simulate / Load Market Data

In [ ]:
# Simulate two-regime GBM for demonstration
np.random.seed(42)
n = 1260  # 5 trading years
dates = pd.date_range('2019-01-02', periods=n, freq='B')

# Regime switching: low-vol vs high-vol periods
regimes = np.zeros(n, dtype=int)
regimes[252:504] = 1   # high vol (e.g. 2020 crash)
regimes[630:756] = 1   # another spike

sigma = np.where(regimes == 0, 0.008, 0.025)
mu = 0.0003
log_rets = np.random.normal(mu, sigma)
close = 100 * np.exp(np.cumsum(log_rets))

df = pd.DataFrame({'close': close, 'log_return': log_rets, 'regime': regimes}, index=dates)
print(df.shape)
df.head()

## 2. Price Series & Log Returns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

axes[0].plot(df.index, df['close'], lw=1, color='steelblue')
axes[0].set_title('Simulated Price Series', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Price ($)')

axes[1].bar(df.index, df['log_return'], width=1, color=np.where(df['log_return'] >= 0, 'green', 'red'), alpha=0.6)
axes[1].set_title('Daily Log Returns', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Log Return')
axes[1].axhline(0, color='black', lw=0.5)

plt.tight_layout()
plt.show()

## 3. Rolling Volatility

In [ ]:
for w in [10, 20, 60]:
    df[f'vol_{w}'] = df['log_return'].rolling(w).std() * np.sqrt(252)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df.index, df['vol_10'], label='10-day vol', alpha=0.7)
ax.plot(df.index, df['vol_20'], label='20-day vol', alpha=0.8, lw=1.5)
ax.plot(df.index, df['vol_60'], label='60-day vol', alpha=0.9, lw=2)

# Shade high-vol regimes
for i in range(len(df)):
    if df['regime'].iloc[i] == 1:
        ax.axvspan(df.index[i], df.index[min(i+1, len(df)-1)], alpha=0.05, color='red')

ax.set_title('Annualised Rolling Volatility with Regime Shading', fontsize=14, fontweight='bold')
ax.set_ylabel('Annualised Volatility')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Return Distribution Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram with normal overlay
r = df['log_return'].dropna()
axes[0].hist(r, bins=80, density=True, alpha=0.6, color='steelblue', label='Empirical')
x = np.linspace(r.min(), r.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, r.mean(), r.std()), 'r-', lw=2, label='Normal fit')
axes[0].set_title('Return Distribution vs Normal', fontsize=13)
axes[0].set_xlabel('Log Return')
axes[0].legend()

# QQ plot
stats.probplot(r, dist='norm', plot=axes[1])
axes[1].set_title('Normal Q-Q Plot of Log Returns', fontsize=13)

kurt = stats.kurtosis(r)
skew = stats.skew(r)
print(f'Kurtosis: {kurt:.3f} | Skewness: {skew:.3f}')
print(f'Fat tails confirmed: {kurt > 1}')

plt.tight_layout()
plt.show()

## 5. Autocorrelation & Volatility Clustering

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(df['log_return'].dropna(), lags=40, ax=axes[0, 0], title='ACF: Log Returns')
plot_pacf(df['log_return'].dropna(), lags=40, ax=axes[0, 1], title='PACF: Log Returns')
plot_acf(df['log_return'].dropna()**2, lags=40, ax=axes[1, 0], title='ACF: Squared Returns (Volatility Clustering)')
plot_acf(df['log_return'].dropna().abs(), lags=40, ax=axes[1, 1], title='ACF: Absolute Returns')

plt.tight_layout()
plt.show()

## 6. Regime Characteristics

In [ ]:
regime_stats = df.groupby('regime')['log_return'].agg(['mean', 'std', 'min', 'max', 'count'])
regime_stats.columns = ['Mean Return', 'Std Dev', 'Min', 'Max', 'Count']
regime_stats['Ann. Vol'] = regime_stats['Std Dev'] * np.sqrt(252)
regime_stats['Ann. Return'] = regime_stats['Mean Return'] * 252
regime_stats.index = ['Low Volatility Regime', 'High Volatility Regime']
print(regime_stats.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for regime, label, color in zip([0, 1], ['Low Vol', 'High Vol'], ['steelblue', 'tomato']):
    r = df[df['regime'] == regime]['log_return']
    axes[0].hist(r, bins=50, density=True, alpha=0.6, label=label, color=color)
axes[0].set_title('Return Distribution by Regime')
axes[0].legend()

axes[1].boxplot([df[df['regime']==0]['log_return'], df[df['regime']==1]['log_return']],
                labels=['Low Vol Regime', 'High Vol Regime'])
axes[1].set_title('Return Spread by Regime')
axes[1].set_ylabel('Log Return')

plt.tight_layout()
plt.show()

## 7. Summary

Key findings from this EDA:

| Observation | Implication |
|---|---|
| **Fat tails** (excess kurtosis > 1) | Normal distribution inadequate; use t-distribution or HMM |
| **Volatility clustering** (ACF of squared returns) | GARCH / HMM regimes appropriate |
| **Two distinct regimes** (low/high vol) | HMM with 2--3 hidden states justified |
| **Negative skewness** | Asymmetric risk; calibrate loss functions accordingly |

These motivate the modelling choices in `src/labeling/build_regimes.py` and `src/training/train_ml.py`.